In [2]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# ================= LOAD ENV =================

load_dotenv()

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
COLLECTION_NAME = os.getenv("QDRANT_COLLECTION")

if not QDRANT_URL:
    raise ValueError("QDRANT_URL manquant dans le .env")

# ================= EMBEDDINGS =================

Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"  # ou "cpu"
)

# ================= CONNEXION QDRANT PROD =================

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

# ================= INDEX VIRTUEL =================

index = VectorStoreIndex.from_vector_store(vector_store)

# ================= TEST RECHERCHE =================

query_engine = index.as_query_engine(
    similarity_top_k=5
)

response = query_engine.query(
    "Résume les données principales stockées dans cette collection"
)

print("Résultat :")
print(response)


Résultat :
Les données principales stockées dans cette collection incluent des décisions de la Commission européenne sur des aides d'État, des informations sur des programmes de soutien à la capacité de retraitement de papier journal, des détails sur la création d'emplois liée à l'utilisation de papier récupéré, des conditions contractuelles entre les parties impliquées, des calculs sur les montants d'aide autorisés, des informations sur la surveillance postérieure à l'octroi de l'aide, des considérations sur la cumulation de différents types d'aides, et des évaluations de la proportionnalité et de la nécessité des aides accordées.
